## PostGIS Integration — Loading Wildfire Data & Building Spatial Queries

**Data:** CAL FIRE historical fire perimeters (`fire24_1.gdb`, 22,810 records) + NASA FIRMS active fire detections (VIIRS S-NPP, California, 2025)

**Database:** PostgreSQL + PostGIS 3.6 (`wildfire_db`, native Windows install)

**What this notebook does:**
- Connects to `wildfire_db` via SQLAlchemy engine and psycopg2 — the two Python interfaces to PostgreSQL
- Loads CAL FIRE fire perimeters from `fire24_1.gdb` into PostGIS table `fire_perimeters` using `gdf.to_postgis()` and creates a GiST spatial index for fast querying
- Downloads NASA FIRMS active fire detection CSV and loads it into PostGIS table `firms_fires`, creating point geometries from lat/lon columns using `ST_MakePoint` and a GiST spatial index
- Writes core spatial queries that will become FastAPI endpoints: `ST_Intersects` (FIRMS detections inside a fire perimeter), `ST_DWithin` (fire perimeters within N km of a coordinate), and top fires by acreage within a radius
- Builds a Python function `get_nearby_fires(lat, lon, radius_km)` that connects to PostGIS, runs a parameterized `ST_DWithin` query, and returns results as a GeoDataFrame via `gpd.read_postgis()`
- Visualizes query results with matplotlib to verify spatial correctness

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

password = os.getenv("DB_PASSWORD")
user = os.getenv("DB_USER")
host = os.getenv("DB_HOST")
dbname = os.getenv("DB_NAME")


In [8]:
import geopandas as gpd
import sqlalchemy as sa
from sqlalchemy import create_engine

In [9]:
# create geodataframe
gdf = gpd.read_file("../data/raw/fire24_1.gdb", layer="firep24_1")

In [10]:
#Create SQLAlchemy engine
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}/{dbname}")


In [ ]:
gdf.to_postgis("fire_perimeters", engine, if_exists="replace")

In [ ]:
#Verify data was written to PostGIS
import psycopg2
conn = psycopg2.connect(host=host, dbname=dbname, user=user, password=password)
cursor = conn.cursor()
cursor.execute("select * from fire_perimeters limit 5;")
results = cursor.fetchall()
print(results)

In [12]:
cursor.execute("CREATE INDEX idx_fire_perimeters_geom ON fire_perimeters USING GIST(geometry);")
conn.commit()
